# 04 · Readmission XGBoost

> **Synthetic data only.** Every dataset used in this notebook is synthetic (Synthea, seed 42) or research-use-only (NIH ChestX-ray14). There is no PHI. None of these models is clinically validated or cleared for patient care.

Build the 30-day readmission feature row with `medflow_ml.features.encounters`, train a small XGBoost model on a synthetic cohort, and inspect feature importance. Illustrative only.

## Encounter featurization
The shared `encounter_feature_row` produces a deterministic vector in `FEATURE_ORDER`, matching serving.

In [ ]:
from __future__ import annotations
from datetime import date
import numpy as np, pandas as pd
from medflow_ml.features.encounters import (
    FEATURE_ORDER, encounter_feature_row, age_band, prior_admission_counts,
)
print('n features:', len(FEATURE_ORDER))
list(FEATURE_ORDER)

## Synthesize a cohort
We draw a reproducible synthetic cohort and featurize each encounter through the shared function.

In [ ]:
rng = np.random.default_rng(42)
N = 400
rows, labels, bands = [], [], []
for _ in range(N):
    age = int(rng.integers(20, 90))
    sex = 'female' if rng.random() < 0.5 else 'male'
    los = float(rng.gamma(2.0, 2.0))
    pa365 = int(rng.poisson(1.2))
    dx = ['I50'] * (rng.random() < 0.3) + ['E11'] * (rng.random() < 0.4)
    rows.append(encounter_feature_row(
        age=age, sex=sex, length_of_stay_days=los,
        prior_admissions_90d=min(pa365, 1),
        prior_admissions_180d=min(pa365, 2),
        prior_admissions_365d=pa365, diagnoses=dx))
    risk = 0.05 + 0.04 * pa365 + 0.03 * len(dx) + 0.01 * (age > 70)
    labels.append(int(rng.random() < min(risk, 0.9)))
    bands.append(age_band(age))
X = pd.DataFrame(rows, columns=list(FEATURE_ORDER))
y = np.array(labels)
X.head()

## Train XGBoost
A small booster with a temporal-style hold-out. The production job adds isotonic calibration and SHAP.

In [ ]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.3, random_state=42)
clf = xgb.XGBClassifier(
    n_estimators=120, max_depth=3, learning_rate=0.1,
    subsample=0.9, objective='binary:logistic', eval_metric='auc',
)
clf.fit(Xtr, ytr)
proba = clf.predict_proba(Xva)[:, 1]

## Evaluate with our metrics
We reuse `medflow_ml.evaluation.metrics` (numpy-only AUROC/AUPRC) so notebooks and jobs report identically.

In [ ]:
from medflow_ml.evaluation.metrics import auroc, auprc
print('AUROC:', round(auroc(yva, proba), 3))
print('AUPRC:', round(auprc(yva, proba), 3))

## Feature importance
Gain-based importance — prior admissions and comorbidity flags should dominate, mirroring LACE/HOSPITAL intuition.

In [ ]:
import matplotlib.pyplot as plt
imp = pd.Series(clf.feature_importances_, index=FEATURE_ORDER).sort_values()
imp.tail(8).plot.barh(figsize=(7, 4), title='XGBoost feature importance')
plt.tight_layout()

### Notes
* Production: `python -m medflow_ml.jobs.train_readmission` registers `readmission-30d`.
* Next: medical imaging (notebook 05).